### FAISS

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [ ]:
# !uv add langchain langchain_openai

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add langchain_teddynote

In [5]:
from langchain_teddynote import logging

logging.langsmith("test0921")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0921


In [ ]:
# !uv add langchain_community

In [11]:
from langchain_community.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)

loader1 = TextLoader("data/nlp-keywords.txt", encoding="utf-8")
loader2 = TextLoader("data/finance-keywords.txt", encoding="utf-8")

split_doc1 = loader1.load_and_split(text_splitter)
split_doc2 = loader2.load_and_split(text_splitter)

len(split_doc1), len(split_doc2)

(11, 6)

In [ ]:
# !uv add faiss-cpu

In [14]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

dimension_size = len(embeddings.embed_query("hello world"))
print(dimension_size)

1536


In [15]:
db = FAISS(
    embedding_function=embeddings,
    index=faiss.IndexFlatL2(dimension_size),
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [16]:
db = FAISS.from_documents(documents=split_doc1, embedding=OpenAIEmbeddings())

In [17]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b'}

In [18]:
db.docstore._dict

{'d2f523f9-d3be-453a-a5cc-db7f9d66ee42': Document(id='d2f523f9-d3be-453a-a5cc-db7f9d66ee42', metadata={'source': 'data/nlp-keywords.txt'}, page_content='Semantic Search\n\n정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.\n예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.\n연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝\n\nEmbedding\n\n정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken\n\n정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.\n예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.\n연관키워드: 토큰화, 자연어 처리, 구문 분석\n\nTokenizer'),
 '12828fdf-ca13-47f8-bfee-1476bb054769': Document(id='12828fdf-ca13-47f8-bfee-1476bb054769', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 토크나이저는 텍스트 데이터를 토큰으로 분할하는 도구입니다. 이는 자연어 처리에서 데이터를 전처리하는 데 사용됩니다.\n예시: "I love programming."이라는 문장을 ["I", "love", "programming", "."]으로 분할합니다.\n연관키워

In [19]:
db2 = FAISS.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding=OpenAIEmbeddings(),
    metadatas=[{"source": "텍스트문서"}, {"source": "텍스트문서"}],
    ids=["doc1", "doc2"],
)

In [20]:
db2.docstore._dict

{'doc1': Document(id='doc1', metadata={'source': '텍스트문서'}, page_content='안녕하세요. 정말 반갑습니다.'),
 'doc2': Document(id='doc2', metadata={'source': '텍스트문서'}, page_content='제 이름은 테디입니다.')}

In [21]:
db.similarity_search("TF IDF 에 대하여 알려줘")

[Document(id='f11238a2-4646-4caf-81d6-7150f570bbf4', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='7daf211f-e83d-41db-b88a-a789dd52c676', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이

In [22]:
db.similarity_search("TF IDF 에 대하여 알려줘", k=2)

[Document(id='f11238a2-4646-4caf-81d6-7150f570bbf4', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='7daf211f-e83d-41db-b88a-a789dd52c676', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이

In [23]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/nlp-keywords.txt"}, k=2
)

[Document(id='f11238a2-4646-4caf-81d6-7150f570bbf4', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning\n\n정의: 딥러닝은 인공신경망을 이용하여 복잡한 문제를 해결하는 머신러닝의 한 분야입니다. 이는 데이터에서 고수준의 표현을 학습하는 데 중점을 둡니다.\n예시: 이미지 인식, 음성 인식, 자연어 처리 등에서 딥러닝 모델이 활용됩니다.\n연관키워드: 인공신경망, 머신러닝, 데이터 분석\n\nSchema\n\n정의: 스키마는 데이터베이스나 파일의 구조를 정의하는 것으로, 데이터가 어떻게 저장되고 조직되는지에 대한 청사진을 제공합니다.\n예시: 관계형 데이터베이스의 테이블 스키마는 열 이름, 데이터 타입, 키 제약 조건 등을 정의합니다.\n연관키워드: 데이터베이스, 데이터 모델링, 데이터 관리\n\nDataFrame'),
 Document(id='7daf211f-e83d-41db-b88a-a789dd52c676', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: 오픈 소스는 소스 코드가 공개되어 누구나 자유롭게 사용, 수정, 배포할 수 있는 소프트웨어를 의미합니다. 이는 협업과 혁신을 촉진하는 데 중요한 역할을 합니다.\n예시: 리눅스 운영 체제는 대표적인 오픈 소스 프로젝트입니다.\n연관키워드: 소프트웨어 개발, 커뮤니티, 기술 협업\n\nStructured Data\n\n정의: 구조화된 데이터는 정해진 형식이나 스키마에 따라 조직된 데이터입니다. 이는 데이터베이

In [24]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/finance-keywords.txt"}, k=2
)

[]

In [25]:
from langchain_core.documents import Document

db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요",
            metadata={"source": "mydata.txt"},
        )
    ],
    ids=["new_doc1"]
)

['new_doc1']

In [26]:
db.similarity_search("안녕하세요", k=1)

[Document(id='new_doc1', metadata={'source': 'mydata.txt'}, page_content='안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요')]

In [27]:
db.add_texts(
    ["이번엔 텍스트 데이터를 추가합니다.", "추가한 2번째 텍스트 데이터 입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["new_doc2", "new_doc3"],
)

['new_doc2', 'new_doc3']

In [28]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [34]:
ids = db.add_texts(
    ["삭제용 데이터를 추가합니다.", "2번째 삭제용 데이터입니다."],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["delete_doc1", "delete_doc2"],
)

In [35]:
print(ids)

['delete_doc1', 'delete_doc2']


In [36]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3',
 14: 'delete_doc1',
 15: 'delete_doc2'}

In [37]:
db.delete(ids)

True

In [38]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [39]:
db.save_local(folder_path="faiss_db", index_name="faiss_index")

In [41]:
loaded_db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

In [42]:
loaded_db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [43]:
db = FAISS.load_local(
    folder_path="faiss_db",
    index_name="faiss_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

In [44]:
db2 = FAISS.from_documents(documents=split_doc2, embedding=OpenAIEmbeddings())

In [45]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3'}

In [46]:
db2.index_to_docstore_id

{0: 'c8e63fbd-8176-4560-9282-1a207e0ba68c',
 1: '8e358d80-8a92-4963-afe4-1905c1ca08aa',
 2: '45b0e322-4585-406b-adf1-a53c60d4776b',
 3: '888788cc-6048-4522-8521-c32eda88aaa2',
 4: '7c60b97a-4c77-4b87-afe1-a26f1aec1372',
 5: '063834f0-91cc-4c77-8ac1-6ef2c7d73549'}

In [47]:
db.merge_from(db2)

In [48]:
db.index_to_docstore_id

{0: 'd2f523f9-d3be-453a-a5cc-db7f9d66ee42',
 1: '12828fdf-ca13-47f8-bfee-1476bb054769',
 2: '28760d6f-1619-4654-8d39-1c523f702a70',
 3: '563d0bdf-266c-46ec-bcad-0268966e23d7',
 4: '7b334247-e848-482d-95a8-a25b736660c0',
 5: '7daf211f-e83d-41db-b88a-a789dd52c676',
 6: 'f11238a2-4646-4caf-81d6-7150f570bbf4',
 7: '826f7e24-4b06-49aa-b836-739dfec0a250',
 8: '90f68cc8-639a-4273-a5d2-4331b07b7d27',
 9: 'cb80fca8-5cca-4566-9759-591988fcd73e',
 10: 'c1859d75-8c67-44b7-8ea5-41cdb90a053b',
 11: 'new_doc1',
 12: 'new_doc2',
 13: 'new_doc3',
 14: 'c8e63fbd-8176-4560-9282-1a207e0ba68c',
 15: '8e358d80-8a92-4963-afe4-1905c1ca08aa',
 16: '45b0e322-4585-406b-adf1-a53c60d4776b',
 17: '888788cc-6048-4522-8521-c32eda88aaa2',
 18: '7c60b97a-4c77-4b87-afe1-a26f1aec1372',
 19: '063834f0-91cc-4c77-8ac1-6ef2c7d73549'}

In [49]:
db = FAISS.from_documents(
    documents=split_doc1 + split_doc2, embedding=OpenAIEmbeddings()
)

In [50]:
retriever = db.as_retriever()

retriever.invoke("Word2Vec 에 대하여 알려줘")

[Document(id='dbd08304-dd69-418f-b99c-78e48dcaba5b', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='c19e6acf-c23c-4aa1-b868-370e0e24196e', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.\n예시: HuggingFace의 Transformers 라이브러리를 사용하여 감정 분석, 텍스트 생성 등의 작업을 수행할 수 있

In [51]:
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25, "fetch_k": 10}
)
retriever.invoke("Word2Vec 에 대하여 알려줘")

[Document(id='dbd08304-dd69-418f-b99c-78e48dcaba5b', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='eeb656b6-c160-4934-aab0-2d2535a81036', metadata={'source': 'data/nlp-keywords.txt'}, page_content='GPT (Generative Pretrained Transformer)\n\n정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 자연스러운 언어를 생성할 수 있습니다.\n예시: 사용자가 제공한 질문에 대해 자세한 답변을 생

In [52]:
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 10})
retriever.invoke("Word2Vec 에 대하여 알려줘")

[Document(id='dbd08304-dd69-418f-b99c-78e48dcaba5b', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source'),
 Document(id='eeb656b6-c160-4934-aab0-2d2535a81036', metadata={'source': 'data/nlp-keywords.txt'}, page_content='GPT (Generative Pretrained Transformer)\n\n정의: GPT는 대규모의 데이터셋으로 사전 훈련된 생성적 언어 모델로, 다양한 텍스트 기반 작업에 활용됩니다. 이는 입력된 텍스트에 기반하여 자연스러운 언어를 생성할 수 있습니다.\n예시: 사용자가 제공한 질문에 대해 자세한 답변을 생

In [54]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.8}
)

retriever.invoke("Word2Vec 에 대하여 알려줘")

[Document(id='dbd08304-dd69-418f-b99c-78e48dcaba5b', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source')]

In [55]:
retriever = db.as_retriever(search_kwargs={"k": 1})

retriever.invoke("Word2Vec 에 대하여 알려줘")

[Document(id='dbd08304-dd69-418f-b99c-78e48dcaba5b', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.\n예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.\n연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)\n\n정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모델은 다양한 자연어 이해 및 생성 작업에 사용됩니다.\n예시: OpenAI의 GPT 시리즈는 대표적인 대규모 언어 모델입니다.\n연관키워드: 자연어 처리, 딥러닝, 텍스트 생성\n\nFAISS (Facebook AI Similarity Search)\n\n정의: FAISS는 페이스북에서 개발한 고속 유사성 검색 라이브러리로, 특히 대규모 벡터 집합에서 유사 벡터를 효과적으로 검색할 수 있도록 설계되었습니다.\n예시: 수백만 개의 이미지 벡터 중에서 비슷한 이미지를 빠르게 찾는 데 FAISS가 사용될 수 있습니다.\n연관키워드: 벡터 검색, 머신러닝, 데이터베이스 최적화\n\nOpen Source')]

In [56]:
retriever = db.as_retriever(
    search_kwargs={"filter": {"source": "data/finance-keywords.txt"}, "k": 2}
)
retriever.invoke("ESG 에 대하여 알려줘")

[Document(id='d69a0938-0940-4745-9697-e06d56908547', metadata={'source': 'data/finance-keywords.txt'}, page_content='정의: ESG는 기업의 환경, 사회, 지배구조 측면을 고려하는 투자 접근 방식입니다.\n예시: S&P 500 ESG 지수는 우수한 ESG 성과를 보이는 기업들로 구성된 지수입니다.\n연관키워드: 지속가능 투자, 기업의 사회적 책임, 윤리 경영\n\nStock Buyback\n\n정의: 자사주 매입은 기업이 자사의 주식을 시장에서 다시 사들이는 것을 말합니다.\n예시: 애플은 S&P 500 기업 중 가장 큰 규모의 자사주 매입 프로그램을 운영하고 있습니다.\n연관키워드: 주주 가치, 자본 관리, 주가 부양\n\nCyclical Stocks\n\n정의: 경기순환주는 경제 상황에 따라 실적이 크게 변동하는 기업의 주식을 말합니다.\n예시: 포드, 제너럴 모터스와 같은 자동차 기업들은 S&P 500에 포함된 대표적인 경기순환주입니다.\n연관키워드: 경제 사이클, 섹터 분석, 투자 타이밍\n\nDefensive Stocks\n\n정의: 방어주는 경기 변동에 상관없이 안정적인 실적을 보이는 기업의 주식을 의미합니다.\n예시: 프록터앤갬블, 존슨앤존슨과 같은 생활필수품 기업들은 S&P 500 내 대표적인 방어주로 꼽힙니다.\n연관키워드: 안정적 수익, 저변동성, 리스크 관리'),
 Document(id='45470733-7010-415a-b983-916759883b0b', metadata={'source': 'data/finance-keywords.txt'}, page_content='정의: 주식 리서치는 기업의 재무 상태, 사업 모델, 경쟁력 등을 분석하여 투자 의사 결정을 돕는 활동입니다.\n예시: 골드만삭스의 애널리스트들이 S&P 500 기업들에 대한 분기별 실적 전망을 발표했습니다.\n연관키워드: 투자 분석, 기업 가치평가, 시장 전망\n\nCorporate 